# ETCC のエネルギースペクトルとバックプロジェクション

図は `figures/` のPNGを参照して表示します。実行例は既定条件の結果です。セルを実行したときの最新結果は、各コードセル直下と `OUT` のPNG・JSONで確認できます。

**読むのは `smile3_reconstruct` で作った再構成 ROOT です。**
この ROOT 内の2つの tree をイベントIDで対応付け、カットごとのスペクトル・方向図・ARMを比較します。

## 1. 入力ファイルと使用するブランチ

### 入力ファイル

**20261116 run03＋run06：12ファイル**

```text
/home/onoda/ericanext_out/20261116_v2/
  etcc_recon_20261116_run03/etcc_recon_20261116_run03.part*.root
  etcc_recon_20261116_run06/etcc_recon_20261116_run06.part*.root
```

run03 は part000〜001 の2ファイル、run06 は part000〜009 の10ファイルです。
線源位置は **(−380, −1090, −1412) mm**。

**20261115 run03：104ファイル**

```text
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/
  etcc_recon_20261115_run03.part*.root
```

part000〜103。`evt_id` と `track_min/max` を追加して作り直した出力を使います。
線源位置は **(0, 0, −152) mm**、天頂角0°です。
実際に選ばれた全ファイル名は、下の設定セルの出力「入力ファイル一覧」で確認できます。

### tree 1：`Reconstruct/single_hit`

保存時点で PSA の1ヒットと TPC のエネルギー条件を満たす事象が、1行に1事象入っています。

- **`evt_id`**：トリガーカウンタ。もう一方の tree との対応付けに使う。
- **`Eg`、`Ke`、`E0` [keV]**：HAのエネルギー、TPCの反跳電子エネルギー、その和。スペクトルに使う。`Ke` は dE/dx 選別、`E0` は方向図・ARMのエネルギー窓にも使う。
- **`has_track`**：飛跡が得られたか。比較の基準は `has_track == 1`。
- **`Pe[3]` [mm]**：散乱点 (X, Y, Z)。Compton cone・ARMの計算と、tree間の対応の確認に使う。
- **`Pg[3]` [mm]**：HAの吸収点 (X, Y, Z)。`Pg − Pe` が散乱ガンマ線の方向。
- **`Vi[3]` [無次元]**：入射ガンマ線の単位方向ベクトル。ETCCの線源方向は **`−Vi`**。
- **`cos_phi` [無次元]**：運動学から求めた散乱角の余弦。Compton cone と ARM に使う。
- **`delta_cos_alpha` [無次元]**：保存された `cos_alpha_geo − cos_alpha_kin`。α整合性カットに使う。
- **`cos_alpha_geo`、`cos_alpha_kin` [無次元]**：読み込むαの元の値。実際の選別に使うのは保存済みの `delta_cos_alpha`。

### tree 2：`TOT_Skewness/tot_skewness`

- **`evt_id`**：`single_hit.evt_id` と同じトリガーカウンタ。
- **`Pe[3]` [mm]**：対応付けた2行で散乱点が一致することを検証する。
- **`Len` [mm]**：飛跡の外接箱の対角線の長さ。dE/dx の帯と飛程上限に使う。
- **`track_min[3]`、`track_max[3]` [mm]**：飛跡点の X・Y・Z の最小値と最大値。飛跡全体への fiducial カットに使う。

線源座標は ROOT のブランチではなく、下の設定の `source_mm` [mm] に実測値を指定します。
エネルギーと検出位置は、上記ブランチに保存された較正済みの値を使います。

### 2つの tree の対応付け

**同じ ROOT ファイルの中で `evt_id` が一致する行を結びます。**
別runの同じイベント番号を混ぜません。飛跡のある行は `Pe` も一致することを確認します。
イベントIDの重複・対応する行の欠落・Peの不一致はエラーで止まります。

解析コードに出てくる `Pe0`・`track_min0` などは読み込み時に作る一時列です。
ROOTに保存されているブランチ名は `Pe`・`track_min` などの3要素配列です。

計算・描画の実装は [compare_etcc_cuts.py](../../scripts/analysis/compare_etcc_cuts.py)。
PyROOT・numpy・matplotlib のあるカーネル（この環境では `/usr/bin/python3`）で実行します。


In [1]:
import sys, glob, json, shlex, subprocess
from pathlib import Path
import numpy as np
import ROOT
from IPython.display import Image, Markdown, HTML, display
from html import escape
ROOT.gROOT.SetBatch(True)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'run_script/run_chain.py').is_file())
SCRIPT = REPO / 'scripts/analysis/compare_etcc_cuts.py'
PYTHON = sys.executable
print('Repository:', REPO)
print('Python:', PYTHON)

Repository: /home/onoda/ANLNext/Erica_git_dev/EricaNext
Python: /usr/bin/python3


## 2. 入力・出力パスを設定する

`DATA_ROOT` は上記の再構成 ROOT の置き場所、`SELECTED_RUNS` は比較する測定です。
`expected_files` は分割ファイルの確認数（この例では12と104）。1ファイル/runなら1、確認しない場合は `None` にします。
図と実行条件のJSONは `OUT / 測定名` に保存します。

- 11/16 の黒い＋：線源 (−380, −1090, −1412) mm を検出器中心 (0, 0, 160) mm から見た方向。
- 11/15 の黒い＋：真上の線源 (0, 0, −152) mm の方向。ARM は各イベントの Pe と実測線源位置から計算する。

入力 ROOT をまだ作っていない場合は、測定に対応した較正設定で先に再構成します。

```bash
python3 run_script/run_chain.py smile3_reconstruct \
  --root-input /path/to/etcc_raw.root -o /path/to/etcc_recon.root
```

生情報の ROOT を作る第1段階は `smile3_etcc_raw`。このノートブックでは再構成済み ROOT を読み、図を描きます。


In [2]:
DATA_ROOT = Path('/home/onoda/ericanext_out')
OUT = DATA_ROOT / 'tutorials/ETCC_CutComparison'
DATASETS = {
    '20261116_run03_06': {
        'patterns': [
            str(DATA_ROOT / '20261116_v2/etcc_recon_20261116_run03/etcc_recon_20261116_run03.part*.root'),
            str(DATA_ROOT / '20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part*.root'),
        ],
        'source_mm': (-380.0, -1090.0, -1412.0),
        'label': '20261116 run03+06 (source zenith 39 deg)',
        'expected_files': 12,
    },
    '20261115_run03': {
        'patterns': [str(DATA_ROOT / '20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part*.root')],
        'source_mm': (0.0, 0.0, -152.0),
        'label': '20261115 run03 (source zenith 0 deg)',
        'expected_files': 104,
    },
}
SELECTED_RUNS = ['20261116_run03_06', '20261115_run03']
INPUTS = {}
for name in SELECTED_RUNS:
    item = DATASETS[name]
    files = sorted(set(f for pattern in item['patterns'] for f in glob.glob(pattern)))
    if not files:
        raise FileNotFoundError(f'{name}: no reconstructed ROOT files')
    expected = item['expected_files']
    if expected is not None and len(files) != expected:
        raise ValueError(f'{name}: expected {expected} files, got {len(files)}; check reconstruction completion')
    if not all(Path(f).is_absolute() for f in files) or not OUT.is_absolute():
        raise ValueError('Use absolute input and output paths')
    INPUTS[name] = files
    print(name, len(files), 'files ->', OUT / name)
    listing = escape('\n'.join(files))
    display(HTML(f'<details><summary>{escape(name)}：入力ファイル一覧（{len(files)}件）</summary><pre>{listing}</pre></details>'))

20261116_run03_06 12 files -> /home/onoda/ericanext_out/tutorials/ETCC_CutComparison/20261116_run03_06


20261115_run03 104 files -> /home/onoda/ericanext_out/tutorials/ETCC_CutComparison/20261115_run03


## 3. カット条件：何を残すか

**現在の既定条件：2気圧、帯の倍率1.0、飛程270 mm未満、α整合性0.5未満。**
**Eg の追加カットと、Ke = 5〜400 keV の範囲カットはありません。**

### 共通の基準

比較する5条件は、すべて次を満たす事象を使います。

- 飛跡あり：`single_hit.has_track == 1`
- 正の反跳電子エネルギー：`single_hit.Ke > 0`
- `Ke`、`E0`、`cos_phi`、`delta_cos_alpha`、`Pe`、`Pg`、`Vi` が有限値。

灰色のスペクトル「All single_hit」は保存された全行を示す参照線です。
比較する5本の線・各方向図には、上の共通基準を適用します。

### fiducial：飛跡全体が検出器の内側にある

`tot_skewness.track_min/max` に、次の3条件を**すべて**要求します。単位は mm。

```text
X：track_min[0] > −148.6 かつ track_max[0] < 148.6
Y：track_min[1] > −148.6 かつ track_max[1] < 148.6
Z：track_min[2] >   15.0 かつ track_max[2] < 312.0
```

外壁・ドリフト領域の両端から5 mm内側という条件です。Peだけが内側でも、飛跡が外へ出ていれば落とします。境界と等しい値も除外します。

### dE/dx：Ke–飛程の帯の中にある

`Ke = single_hit.Ke` [keV]、`L = tot_skewness.Len` [mm] として、帯を次で定義します。

```python
L_min = 913.9 * (Ke / 1000.0)**2.0 + 2.2      # mm
L_max = 1975.9 * (Ke / 1000.0)**1.4 + 7.0     # mm
```

残す条件は次の**両方**です。

```text
L_min < L < L_max
L < 270 mm
```

SMILE-2+と同じ2気圧なので、帯と270 mmの上限をそのまま使います。180 mmへの短縮は行いません。
Keには5 keVの下限も400 keVの上限も付けません（共通基準の Ke > 0 は維持）。
この帯はSMILE-3の実測で最適化したものではありません。

### α整合性：運動学と飛跡から求めた値が近い

`single_hit.delta_cos_alpha` に次を要求します。

```text
abs(delta_cos_alpha) < 0.5
```

これは余弦の差の上限で、角度の0.5°ではありません。等号は除外します。

### 5条件の比較

図の順番と `CUT_TO_SHOW` の番号は次の対応です。

- **0：共通基準のみ**
- **1：共通基準 ＋ fiducial**
- **2：共通基準 ＋ dE/dx**
- **3：共通基準 ＋ fiducial ＋ dE/dx**
- **4：共通基準 ＋ fiducial ＋ dE/dx ＋ α整合性**

### エネルギー窓を使う図

**エネルギースペクトルには E0 窓をかけません。**
方向図・ARM・dE/dx確認図の下段だけ、上の条件に次を加えます。

```text
630 keV <= single_hit.E0 <= 700 keV
abs(single_hit.cos_phi) <= 1
```

### 下の設定セルの読み方

- `MAP_E0_KEV = (630.0, 700.0)`：方向図・ARMに使う E0 の下限・上限 [keV]。
- `ALPHA_MAX_DELTA_COS = 0.5`：α整合性の上限 [無次元]。
- `FIDUCIAL_MARGIN_MM = 5.0`：外壁・ドリフト領域の両端からの余白 [mm]。
- `LENGTH_SCALE = 1.0`：SMILE-2+の帯と飛程上限に掛ける倍率 [無次元]。現在は同じ2気圧なので1.0。
- `SKY_BINS = 24`：方向図の1軸のビン数。天頂角半径図では7.5°幅。Lambert図の軸はdegree-equivalent。

値を変更した場合は「4. 描画する」以降を実行し直します。実行時の条件は各出力先の `selection_summary.json` に保存されます。


In [3]:
MAP_E0_KEV = (630.0, 700.0)
ALPHA_MAX_DELTA_COS = 0.5
FIDUCIAL_MARGIN_MM = 5.0
LENGTH_SCALE = 1.0
SKY_BINS = 24

## 4. 描画する

共通の解析スクリプトを実行します。再構成・較正は実行しません。
同じ出力先のPNGとJSONは、指定した条件で更新します。
入力の不整合、重複イベントID、Peの不一致はエラーで止まります。

In [4]:
run_logs = []
for name in SELECTED_RUNS:
    item = DATASETS[name]
    cmd = [PYTHON, str(SCRIPT), *INPUTS[name], '--out', str(OUT / name),
           '--src', *map(str, item['source_mm']), '--label', item['label'],
           '--emin', str(MAP_E0_KEV[0]), '--emax', str(MAP_E0_KEV[1]),
           '--alpha', str(ALPHA_MAX_DELTA_COS), '--margin', str(FIDUCIAL_MARGIN_MM),
           '--len-scale', str(LENGTH_SCALE), '--bins', str(SKY_BINS)]
    result = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode:
        print(result.stdout)
        result.check_returncode()
    run_logs.append(shlex.join(cmd) + '\n' + result.stdout)
SUMMARIES = {name: json.loads((OUT / name / 'selection_summary.json').read_text())
             for name in SELECTED_RUNS}
log_text = escape('\n'.join(run_logs))
display(HTML(f'<details><summary>実行ログ（{len(SELECTED_RUNS)}ラン）</summary><pre>{log_text}</pre></details>'))

実行ログ（2ラン） /usr/bin/python3 /home/onoda/ANLNext/Erica_git_dev/EricaNext/scripts/analysis/compare_etcc_cuts.py /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run03/etcc_recon_20261116_run03.part000.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run03/etcc_recon_20261116_run03.part001.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part000.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part001.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part002.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part003.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part004.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part005.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part006.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part007.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part008.root /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part009.root --out /home/onoda/ericanext_out/tutorials/ETCC_CutComparison/20261116_run03_06 --src -380.0 -1090.0 -1412.0 --label '20261116 run03+06 (source zenith 39 deg)' --emin 630.0 --emax 700.0 --alpha 0.5 --margin 5.0 --len-scale 1.0 --bins 24
/usr/bin/python3 /home/onoda/ANLNext/Erica_git_dev/EricaNext/scripts/analysis/compare_etcc_cuts.py /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part000.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part001.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part002.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part003.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part004.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part005.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part006.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part007.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part008.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part009.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part010.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part011.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part012.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part013.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part014.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part015.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part016.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part017.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part018.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part019.root /home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part020.root /home/onoda/ericanext_out/20261

## 5. 残存イベント数

スペクトルの件数、E0窓と $|\cos\phi|\le1$ を満たす件数、ETCC方向図の上半球に入る件数を区別します。
ARM の中央値・四分位範囲 (IQR) は度で表示します。IQR はフィットした角度分解能ではありません。
入力ファイル・選別条件・件数は `selection_summary.json` に残ります。

In [5]:
SUMMARIES = {name: json.loads((OUT / name / 'selection_summary.json').read_text())
             for name in SELECTED_RUNS}
cut_labels = ['共通基準のみ', '基準 + fiducial', '基準 + dE/dx',
              '基準 + fiducial + dE/dx', '基準 + fiducial + dE/dx + α整合性']
for name, result in SUMMARIES.items():
    heads = ['条件', 'スペクトル', 'E0窓', '上半球', 'ARM中央値 [deg]', 'IQR [deg]']
    header = ''.join(f'<th style="border:1px solid #bbb;padding:6px">{escape(h)}</th>' for h in heads)
    rows = []
    for i, row in enumerate(result['cuts']):
        med, iqr = row['ARM_median_deg'], row['ARM_IQR_deg']
        values = [f'{i}: {cut_labels[i]}', f"{row['spectrum_events']:,}",
                  f"{row['E0_window_events']:,}", f"{row['upper_hemisphere_events']:,}",
                  '—' if med is None else f'{med:.1f}', '—' if iqr is None else f'{iqr:.1f}']
        cells = ''.join(f'<td style="border:1px solid #bbb;padding:6px;text-align:{"left" if j == 0 else "right"}">{escape(v)}</td>'
                        for j, v in enumerate(values))
        rows.append(f'<tr>{cells}</tr>')
    display(HTML(f'<table class="etcc-counts" style="border-collapse:collapse">'
                 f'<caption style="text-align:left;font-weight:bold">{escape(name)}</caption>'
                 f'<thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table>'))

条件,スペクトル,E0窓,上半球,ARM中央値 [deg],IQR [deg]
0: 共通基準のみ,"381,229","23,114","20,026",37.2,49.4
1: 基準 + fiducial,"155,395","11,849","10,756",33.9,46.5
2: 基準 + dE/dx,"82,747","8,597","7,849",29.6,46.7
3: 基準 + fiducial + dE/dx,"59,553","6,494","6,082",29.1,44.1
4: 基準 + fiducial + dE/dx + α整合性,"32,102","3,820","3,645",23.0,38.2


条件,スペクトル,E0窓,上半球,ARM中央値 [deg],IQR [deg]
0: 共通基準のみ,"2,235,404","247,040","226,773",43.3,40.7
1: 基準 + fiducial,"1,311,773","142,710","135,402",39.2,38.9
2: 基準 + dE/dx,"826,411","95,488","88,010",38.9,41.5
3: 基準 + fiducial + dE/dx,"624,444","71,919","67,670",36.9,40.0
4: 基準 + fiducial + dE/dx + α整合性,"325,600","41,595","39,830",30.2,37.7


## 6. エネルギースペクトル

左から Eg、Ke、E0。5 keV/bin、縦軸対数、規格化しない実数です。
**Eg カットも E0 窓もかけていません**。Ke = 5–400 keV の範囲カットはありません。Ke の表示範囲は0–600 keVで、選別条件とは別です。

In [6]:
for name in SELECTED_RUNS:
    display(Markdown(f'**{name}**'))
    display(Image(filename=str(OUT / name / 'energy_cut_comparison.png')))

**20261116_run03_06**

**20261115_run03**

**実行例：エネルギースペクトル**（このノートブックの既定条件、2気圧・倍率1.0・Ke範囲カットなし）

**20261116_run03_06**

![20261116_run03_06 エネルギースペクトル](figures/20261116_run03_06_energy.png)

**20261115_run03**

![20261115_run03 エネルギースペクトル](figures/20261115_run03_energy.png)

## 6.1. dE/dx カットを Ke–飛程の分布で確認する

横軸 Ke [keV]、縦軸飛程 Len [mm] の2次元分布です。dE/dx の選別に使う Ke–Len の帯を示し、Ke/Len を直接描く図とは区別します。
左から追加カットなし、fiducial のみ、fiducial＋dE/dx。上段は E0 窓なし、下段は方向図と同じ E0 窓と運動学の条件です。Eg の追加カットはありません。

- 灰色の点線: SMILE-2+ の元の帯。黒い実線: LENGTH_SCALE 倍して実際に使う帯。既定1.0では両者は重なります。
- 黒い破線: Len の上限（既定270 mm）。Ke = 5–400 keV の境界・範囲選別は使いません。
- 色は生の件数を共通の対数スケールで表示し、空ビンはマスクします。4 keV × 3 mm/bin。
- 表示範囲は Ke = 0–600 keV、Len = 0–300 mm。N は選択された全件数、shown は表示範囲内の件数です。

分布と境界を見て、帯の位置・fiducial で落ちる集団・残る集団を確認してください。

In [7]:
for name in SELECTED_RUNS:
    display(Markdown(f'**{name}: Ke–track length**'))
    display(Image(filename=str(OUT / name / 'dedx_cut_comparison.png')))

**20261116_run03_06: Ke–track length**

**20261115_run03: Ke–track length**

**実行例：dE/dx確認図**（このノートブックの既定条件、2気圧・倍率1.0・Ke範囲カットなし）

**20261116_run03_06**

![20261116_run03_06 dE/dx確認図](figures/20261116_run03_06_dedx.png)

**20261115_run03**

![20261115_run03 dE/dx確認図](figures/20261115_run03_dedx.png)

## 7. 各カット後のバックプロジェクションと ARM

各段は左から Compton cone（飛跡方向を使わない）、ETCC（飛跡方向を使う）、ARM。
上から追加カットなし、fiducial のみ、dE/dx のみ、両方、両方＋α整合性です。

- 方向図の半径は天頂角 $\theta$、横軸 $\theta\cos\varphi$、縦軸 $\theta\sin\varphi$ [deg]。
  天頂は −Z、+X は東、+Y は南。黒い＋は実測線源の方向です。
- ETCC の線源方向は $-V_i$。Compton cone は $P_e-P_g$ を軸として散乱角 $\phi$ の円周を360点で積分し、1点の重みを1/360にします。
  上半球で切るので cone の積算値はイベント数そのものとは一致しません。
- ARM は $\angle(P_e-P_{src},P_g-P_e)-\phi$ [deg]。近距離の線源にも各イベントの Pe を使います。
- 生カウントの方向図であり、等立体角ビンへの変換・検出効率の補正は行っていません。

`COMMON_COLOR_SCALE=False` は形を見比べるため各段で色スケールを合わせます。
`True` はカット間で同じ色スケールを使い、絶対件数の減少を見比べます。

In [8]:
COMMON_COLOR_SCALE = False
map_file = ('backprojection_cut_comparison.png' if COMMON_COLOR_SCALE
            else 'backprojection_cut_comparison_autoscale.png')
for name in SELECTED_RUNS:
    display(Markdown(f'**{name}**'))
    display(Image(filename=str(OUT / name / map_file)))

**20261116_run03_06**

**20261115_run03**

**実行例：各カット後のバックプロジェクション**（このノートブックの既定条件、2気圧・倍率1.0・Ke範囲カットなし）

**20261116_run03_06**

![20261116_run03_06 各カット後のバックプロジェクション](figures/20261116_run03_06_backprojection.png)

**20261115_run03**

![20261115_run03 各カット後のバックプロジェクション](figures/20261115_run03_backprojection.png)

## 8. 1条件だけ大きく表示する

`CUT_TO_SHOW` は0:追加なし、1:fiducial、2:dE/dx、3:両方、4:両方＋α整合性。
単独図の色スケールは各条件に合わせています。

In [9]:
CUT_TO_SHOW = 4
for name in SELECTED_RUNS:
    display(Markdown(f'**{name}**'))
    display(Image(filename=str(OUT / name / f'backprojection_{CUT_TO_SHOW:02d}.png')))

**20261116_run03_06**

**20261115_run03**

**実行例：fiducial＋dE/dx＋α整合性の方向図**（このノートブックの既定条件、2気圧・倍率1.0・Ke範囲カットなし）

**20261116_run03_06**

![20261116_run03_06 fiducial＋dE/dx＋α整合性の方向図](figures/20261116_run03_06_selected.png)

**20261115_run03**

![20261115_run03 fiducial＋dE/dx＋α整合性の方向図](figures/20261115_run03_selected.png)

## 解釈するときの確認点

TPC と PSA の座標系は一致すると仮定しています。ピクセルの向き・飛跡方向・較正の妥当性は別に確認が必要です。
カットを加えて件数が減ったことと、ARM の偏りが改善したことを分けて確認してください。
方向図が中心に集まるだけでは正しい再構成と判断せず、実測線源方向、ARM、イベント表示も併せて見ます。
低統計では `SKY_BINS` を小さくし、平滑化で構造を付け加えずにビンをまとめます。

1事象の飛跡を確認する場合は `scripts/analysis/plot_etcc_event.py` を使います。
`--min-len` / `--max-len` [mm]、`--top` [events]、`--order points|length` で自動選択を変更できます。
入力 ROOT は `TOT_Skewness.flag_save_cloud=true` で保存した点群が必要です。

背景だけでも ARM の中央値が正に偏る場合があります。中央値やピークの位置だけで線源信号の有無を判断せず、真の線源位置と回転した偽線源位置の差、または測定時間などで規格化した線源あり・なしの差を確認します。このノートブックは背景を差し引く前の分布です。

今回の較正には次の制約があります（2026-11-17時点）。TPC のゲインマップは576組中230組のみ較正済みで、未較正の寄与は0です。全体の二次補正は負の寄与を除く修正前のスペクトルから決めています。ドリフト速度は run03 の上端の鈍い分布から得た暫定値（品質判定NG）。PSA は温度補正なしで、Cs と Co の較正ランに1〜2 °Cの差があります。これらを含めて較正・背景を確認するまで、分布の幅を装置の性能として扱わないでください。


## 9. Lambert 正積投影でも比較する

ここまでの図は、最初に比較した図と同じ「天頂角を半径にする図」です。
Claude の引き継ぎスクリプトの最新版と同じ Lambert 正積投影も、`--projection lambert` で描けます。

半径は $r=2\sin(\theta/2)$。軸の数値は $r\times180/\pi$（角度そのものではなく degree-equivalent）で、上半球の半径は約81.03です。
同心円は天頂角30、60、90°を示します。平面内の等しい面積が等しい立体角に対応し、天頂角半径図の面積による偏りを避けられます。
円周を横切る部分ビンの面積と検出効率は補正していません。

`MAKE_LAMBERT=False` にすれば、この追加描画は省略できます。カット、入力事象、E0窓は変更しません。
出力は各ランの `lambert/` に置き、元の図も残します。

In [10]:
MAKE_LAMBERT = True
if MAKE_LAMBERT:
    run_logs = []
    for name in SELECTED_RUNS:
        item = DATASETS[name]
        dest = OUT / name / 'lambert'
        cmd = [PYTHON, str(SCRIPT), *INPUTS[name], '--out', str(dest),
               '--src', *map(str, item['source_mm']), '--label', item['label'],
               '--emin', str(MAP_E0_KEV[0]), '--emax', str(MAP_E0_KEV[1]),
               '--alpha', str(ALPHA_MAX_DELTA_COS), '--margin', str(FIDUCIAL_MARGIN_MM),
               '--len-scale', str(LENGTH_SCALE), '--bins', str(SKY_BINS),
               '--projection', 'lambert']
        result = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        if result.returncode:
            print(result.stdout)
            result.check_returncode()
        run_logs.append(shlex.join(cmd) + '\n' + result.stdout)
        lam = json.loads((dest / 'selection_summary.json').read_text())
        for original, projected in zip(SUMMARIES[name]['cuts'], lam['cuts']):
            for key in ['spectrum_events', 'E0_window_events', 'upper_hemisphere_events']:
                assert original[key] == projected[key], (name, key)
        display(Markdown(f'**{name}: Lambert equal-area**'))
        display(Image(filename=str(dest / 'backprojection_cut_comparison_autoscale.png')))
    log_text = escape('\n'.join(run_logs))
    display(HTML(f'<details><summary>Lambert実行ログ（{len(SELECTED_RUNS)}ラン）</summary><pre>{log_text}</pre></details>'))

**20261116_run03_06: Lambert equal-area**

**20261115_run03: Lambert equal-area**

実行ログ（2ラン） /home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run03/etcc_recon_20261116_run03.part000.root 40256 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run03/etcc_recon_20261116_run03.part001.root 39934 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part000.root 56430 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part001.root 56457 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part002.root 46927 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part003.root 16795 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part004.root 18608 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part005.root 18682 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part006.root 21992 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part007.root 28108 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part008.root 26453 single_hit; join OK
/home/onoda/ericanext_out/20261116_v2/etcc_recon_20261116_run06/etcc_recon_20261116_run06.part009.root 14028 single_hit; join OK
[
 {
 "cut": "Track, no additional cut",
 "spectrum_events": 381229,
 "E0_window_events": 23114,
 "upper_hemisphere_events": 20026,
 "ARM_median_deg": 37.174920477032,
 "ARM_IQR_deg": 49.369231809555004
 },
 {
 "cut": "Fiducial only",
 "spectrum_events": 155395,
 "E0_window_events": 11849,
 "upper_hemisphere_events": 10756,
 "ARM_median_deg": 33.921944308369575,
 "ARM_IQR_deg": 46.468814277185736
 },
 {
 "cut": "dE/dx only",
 "spectrum_events": 82747,
 "E0_window_events": 8597,
 "upper_hemisphere_events": 7849,
 "ARM_median_deg": 29.616572342377417,
 "ARM_IQR_deg": 46.73409744042221
 },
 {
 "cut": "Fiducial + dE/dx",
 "spectrum_events": 59553,
 "E0_window_events": 6494,
 "upper_hemisphere_events": 6082,
 "ARM_median_deg": 29.07166626472621,
 "ARM_IQR_deg": 44.07837551323377
 },
 {
 "cut": "Fiducial + dE/dx + |delta cos alpha| < 0.5",
 "spectrum_events": 32102,
 "E0_window_events": 3820,
 "upper_hemisphere_events": 3645,
 "ARM_median_deg": 22.964599422544673,
 "ARM_IQR_deg": 38.19960523256182
 }
]
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part000.root 13555 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part001.root 13127 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part002.root 20723 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part003.root 20190 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part004.root 21669 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part005.root 16765 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part006.root 20911 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part007.root 20712 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part008.root 18253 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon_20261115_run03.part009.root 17575 single_hit; join OK
/home/onoda/ericanext_out/20261115_selected/etcc_recon_20261115_run03/etcc_recon

**実行例：Lambert 正積図**（このノートブックの既定条件、2気圧・倍率1.0・Ke範囲カットなし）

**20261116_run03_06**

![20261116_run03_06 Lambert 正積図](figures/20261116_run03_06_lambert.png)

**20261115_run03**

![20261115_run03 Lambert 正積図](figures/20261115_run03_lambert.png)